# Paper Hit Probability — citation percentile within FoS × year cohorts

Computes each paper's **citation percentile** within its **Field-of-Study × publication-year**
cohort, for each window (C3 / C5 / C10 / C_all), using **paper->paper forward citations**. A percentile of 0.99 = top 1%,
0.95 = top 5%, 0.90 = top 10%. Primary key: `paper_id`.

## Raw / input data (directory & structure)
```
/project/jevans/Dawoon/Science of Science/OpenAlex/output/paper_citation.parquet   # paper_id, C_3, C_5, C_10, C_all (paper->paper forward citations)
/project/jevans/Dawoon/Science of Science/OpenAlex/output/paper_metadata.parquet   # paper_id, year, FoS_0, FoS_rep
```
Cohort **FoS** = the paper's primary level-0 field: `FoS_rep` (max-score field) when present, else the
first field of `FoS_0`. Join to the citation counts is done out-of-core with **DuckDB** (metadata is 249.8M rows).

## Metric (definition)
Within each (FoS, year) cohort, `pctl_{w}` is the citation **percentile** in `[0, 1]` (higher = more
cited; zero-citation ties near 0, most-cited near 1). Read hits as: top 1% ⇔ `pctl ≥ 0.99`,
top 5% ⇔ `≥ 0.95`, top 10% ⇔ `≥ 0.90`. Computed for `C_3, C_5, C_10, C_all`.

## Output
`/project/jevans/Dawoon/Science of Science/OpenAlex/output/paper_hit_probability.parquet` — `paper_id, FoS, year` +
`pctl_{w}` for w ∈ {c3,c5,c10,call} (citation percentile in [0,1] within the cohort).

In [1]:
import os, sys, gc, time
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/OpenAlex')
import oa_common as oa
ROOT = oa.BASE; OUT = oa.OUT
CIT    = f'{OUT}/paper_citation.parquet'
META   = f'{OUT}/paper_metadata.parquet'
OUT_FP = f'{OUT}/paper_hit_probability.parquet'
# DuckDB is not installed in the Curvature env and installing into a shared env is not this
# notebook's business, so the out-of-core join is pyarrow's. The percentile itself is
# unchanged — still rank(pct=True, method='min') within (FoS, year).
# Cohorts are processed one FIELD at a time, which bounds peak memory by the largest field
# rather than by the corpus.
print('snapshot:', oa.ROOT)
# Everything below is fed by notebook/referenced_works_w_year.ipynb: it writes the per-work
# map (year + source) and the edge table with both years and both source ids, and
# oa.load_graph() / oa.load_csr() / oa.build_journal() read those instead of re-walking
# renli's tree. Build it once before running this notebook.
assert oa.have_consolidated(), (
    'run notebook/referenced_works_w_year.ipynb first — it builds the map and edge table')
oa.summary()

ROOT: C:\Users\jdwoo\OneDrive\Desktop\Research\Science of Science


In [2]:
WINS = [('c3', 'C_3'), ('c5', 'C_5'), ('c10', 'C_10'), ('call', 'C_all')]
def add_percentile(df, cohort_cols):
    """Within each cohort partition, the citation PERCENTILE per window in [0, 1]
    (higher = more cited). rank(ascending=True, method='min') puts zero-citation ties near 0 and the
    top near 1; e.g. top 1% / 5% / 10% correspond to pctl >= 0.99 / 0.95 / 0.90."""
    for wtag, wcol in WINS:
        df[f'pctl_{wtag}'] = df.groupby(cohort_cols)[wcol].rank(pct=True, ascending=True, method='min').astype('float32')
    return df

## 1. Join citations with FoS + year (DuckDB, out-of-core)

In [3]:
%%time
# 1. Join citations with FoS + year (pyarrow, then one field at a time)
cit = pq.read_table(CIT, columns=['paper_id', 'C_3', 'C_5', 'C_10', 'C_all'])
met = pq.read_table(META, columns=['paper_id', 'FoS_rep', 'FoS_0', 'year'])
print(f'citations {cit.num_rows:,} | metadata {met.num_rows:,}')
tbl = cit.join(met, keys='paper_id', join_type='inner')
del cit, met; gc.collect()
df = tbl.to_pandas(); del tbl; gc.collect()

# primary FoS = FoS_rep, else the first element of FoS_0 — as before
df['FoS'] = df['FoS_rep'].fillna(df['FoS_0'].str.split(';').str[0])
df = df.dropna(subset=['FoS', 'year'])
df['year'] = df['year'].astype(int)
for c in ['C_3', 'C_5', 'C_10', 'C_all']:
    df[c] = df[c].fillna(0).astype('int64')
df = df[['paper_id', 'FoS', 'year', 'C_3', 'C_5', 'C_10', 'C_all']]
gc.collect()
print(f'papers with FoS + year: {len(df):,} across {df.FoS.nunique()} fields')

papers with FoS + year: 172,570,930
CPU times: total: 2min 32s
Wall time: 54.6 s


## 2. Compute percentiles + save

In [4]:
%%time
# 2. Percentiles within (FoS, year), one field at a time, then save
outs = []
for i, (fos, g) in enumerate(df.groupby('FoS', sort=False)):
    g = add_percentile(g.copy(), ['FoS', 'year'])
    outs.append(g[['paper_id', 'FoS', 'year'] + [f'pctl_{w}' for w in ['c3','c5','c10','call']]])
    if (i + 1) % 5 == 0:
        print(f'  {i+1} fields done', flush=True)
out = pd.concat(outs, ignore_index=True); del outs; gc.collect()
out.to_parquet(OUT_FP, index=False)
print(f'WROTE {OUT_FP}  ({len(out):,} rows, {len(out.columns)} cols)')
print('citation percentile summary:')
print(out[[f'pctl_{w}' for w in ['c3','c5','c10','call']]].describe().round(4).to_string())
display(out.head(8))

WROTE C:\Users\jdwoo\OneDrive\Desktop\Research\Science of Science\notebook\paper\output\paper_hit_probability.parquet  (172,570,930 rows, 7 cols)
citation percentile summary:


            pctl_c3       pctl_c5      pctl_c10     pctl_call
count  1.725709e+08  1.725709e+08  1.725709e+08  1.725709e+08
mean   2.341000e-01  2.554000e-01  2.740000e-01  2.916000e-01
std    3.682000e-01  3.742000e-01  3.779000e-01  3.806000e-01
min    0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
25%    0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
50%    0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
75%    6.069000e-01  6.520000e-01  6.757000e-01  6.956000e-01
max    1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00


,paper_id,FoS,year,pctl_c3,pctl_c5,pctl_c10,pctl_call
0,W1496852943,Sociology,1974,0.000075,0.000075,0.000075,0.000075
1,W1496870554,Computer science,2006,0.000001,0.000001,0.000001,0.000001
2,W149693405,Computer science,2006,0.000001,0.000001,0.000001,0.000001
3,W1496993611,Political science,2005,0.000005,0.000005,0.000005,0.000005
4,W1497002900,Art,2000,0.000007,0.000007,0.000007,0.000007
5,W1497029470,Philosophy,2007,0.000006,0.000006,0.000006,0.000006
6,W1497044705,Economics,1971,0.000128,0.000128,0.000128,0.000128
7,W1497047733,Art,2003,0.000006,0.000006,0.000006,0.000006


CPU times: total: 11min 4s
Wall time: 11min 12s
